# 11 Static fOU Convergence Calibration

Label first passages directly from observed spread paths. Option exits do not determine forecast success or censoring. No backtest is rerun.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_session import NotebookSession

session = NotebookSession.active()
cfg = session.config
RUN_DIR = session.run
session.begin('11_calibration', ['10_equilibrium'])


## 2. Load saved forecasts and label paths

The event starts at the signal date and ends H observed sessions later. Early option expiry alone is not censoring.


In [ ]:
from src.forecast_calibration import label_forecasts, summarize_forecasts
forecasts = session.frame('forecasts')
test_prices = session.frame('test_prices')
trades = session.frame('trades')
calibration = label_forecasts(forecasts, test_prices)
session.save('forecast_calibration', calibration)
summary = summarize_forecasts(calibration)
session.json('calibration_summary', summary)
display(pd.Series(summary))


## 3. Traded forecast subset

Headline rates use complete forecast windows; this avoids keeping early successes while dropping incomplete failures. Also report the separate traded-forecast population.


In [ ]:
traded_forecasts = forecasts.loc[forecasts.forecast_id.isin(trades.forecast_id)] if not trades.empty else forecasts.iloc[:0]
traded_calibration = label_forecasts(traded_forecasts, test_prices)
session.save('traded_forecast_calibration', traded_calibration)
traded_summary = summarize_forecasts(traded_calibration)
session.json('traded_forecast_calibration_summary', traded_summary)
display(pd.Series(traded_summary))
if not calibration.empty:
    display(calibration.event_status.value_counts())


## 4. Calibration by horizon and exit economics

Overlapping forecasts are not independent binomial trials. Mean predicted probability at the maximum horizon is not the selected-horizon target.


In [ ]:
if not calibration.empty:
    complete = calibration.loc[calibration.complete_horizon_observed].copy()
    if not complete.empty:
        complete['horizon_bin'] = pd.cut(complete.convergence_horizon_trading_days, bins=[0, 5, 20, 63, 126, np.inf])
        buckets = complete.groupby('horizon_bin', observed=True).agg(n_forecasts=('pair', 'size'),
            mean_prediction=('probability_at_selected_horizon', 'mean'), observed_rate=('realized_within_selected_horizon', 'mean'))
        buckets.index = buckets.index.astype(str)
        session.save('calibration_horizon_buckets', buckets)
        display(buckets)
        buckets[['mean_prediction', 'observed_rate']].plot.bar(figsize=(10, 4), title='Selected-horizon forecast calibration')
        plt.tight_layout()
        plt.show()
if not trades.empty:
    display(trades.groupby('exit_reason').agg(n_trades=('pnl', 'size'), total_pnl=('pnl', 'sum'), mean_return=('trade_return', 'mean')))


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
session.finish('11_calibration')
